Nome: PORTO ALEGRE - JARDIM BOTANICO
Codigo Estacao: A801
Latitude: -30.05361111
Longitude: -51.17472221
Altitude: 41.18
Situacao: Operante
Data Inicial: 2001-01-01
Data Final: 2026-07-17
Periodicidade da Medicao: Diaria

In [1]:
#DISCLAIMER: The entire notebook was reviewed by AI to ensure its optimization upon release (hence the Portuguese variable names). Some parts of the code were also AI assisted

In [2]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [3]:
# Loading the CSV
df = pd.read_csv('dados-jardim-botanico.csv', sep=';', decimal=',', encoding='utf-8')
df['date'] = pd.to_datetime(df['Data Medicao'])
df = df.sort_values('date').reset_index(drop=True)

In [4]:
df = df.drop(columns=['Unnamed: 11'])

In [5]:
import pandas as pd
import numpy as np

# precipitation column
col_precip = 'PRECIPITACAO TOTAL, DIARIO (AUT)(mm)'

def classificar_tempo(precip):
    if pd.isna(precip):
        return np.nan
    if precip == 0:
        return 'no_rain'
    elif precip <= 2.5:
        return 'light_rain'
    elif precip <= 10:
        return 'moderate_rain'
    elif precip <= 50:
        return 'heavy_rain'
    else:
        return 'extreme_rain'

df['tipo_climatico'] = df[col_precip].apply(classificar_tempo)

In [6]:
dias_reais = set(df['date'])
df['month'] = df['date'].dt.month
df = df.set_index('date').asfreq('D')  # create lines for the days missing
df = df.reset_index()

# jointing 'extreme rain' with 'heavy rain' (I noticed they were very rare, decided to remain the same code in the cell before though)
df['tipo_climatico'] = df['tipo_climatico'].replace('extreme_rain', 'heavy_rain')
df['temp_max_mean_3d'] = df['TEMPERATURA MAXIMA, DIARIA (AUT)(°C)'].rolling(window=3).mean()
df['precipitation_sum_3d'] = df['PRECIPITACAO TOTAL, DIARIO (AUT)(mm)'].rolling(window=3).sum()
df['wind_mean_3d'] = df['VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)'].rolling(window=3).mean()
df['target_tomorrow'] = df['tipo_climatico'].shift(-1)  

df = df.set_index('date').asfreq('D').reset_index()
df['dia_fantasma'] = ~df['date'].isin(dias_reais)

df = df[~df['dia_fantasma']]
df = df.dropna(subset=['target_tomorrow'])

In [7]:
split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

In [8]:
features = ['PRECIPITACAO TOTAL, DIARIO (AUT)(mm)',
       'PRESSAO ATMOSFERICA MEDIA DIARIA (AUT)(mB)',
       'TEMPERATURA DO PONTO DE ORVALHO MEDIA DIARIA (AUT)(°C)',
       'TEMPERATURA MAXIMA, DIARIA (AUT)(°C)',
       'TEMPERATURA MEDIA, DIARIA (AUT)(°C)',
       'TEMPERATURA MINIMA, DIARIA (AUT)(°C)',
       'UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)',
       'UMIDADE RELATIVA DO AR, MINIMA DIARIA (AUT)(%)',
       'VENTO, RAJADA MAXIMA DIARIA (AUT)(m/s)',
       'VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)',
       'month', 'tipo_climatico', 'temp_max_mean_3d', 'precipitation_sum_3d',
       'wind_mean_3d']
target = 'target_tomorrow'

In [9]:
X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

In [10]:
num_features = ['PRECIPITACAO TOTAL, DIARIO (AUT)(mm)',
       'PRESSAO ATMOSFERICA MEDIA DIARIA (AUT)(mB)',
       'TEMPERATURA DO PONTO DE ORVALHO MEDIA DIARIA (AUT)(°C)',
       'TEMPERATURA MAXIMA, DIARIA (AUT)(°C)',
       'TEMPERATURA MEDIA, DIARIA (AUT)(°C)',
       'TEMPERATURA MINIMA, DIARIA (AUT)(°C)',
       'UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)',
       'UMIDADE RELATIVA DO AR, MINIMA DIARIA (AUT)(%)',
       'VENTO, RAJADA MAXIMA DIARIA (AUT)(m/s)',
       'VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)',
       'month','temp_max_mean_3d', 'precipitation_sum_3d',
       'wind_mean_3d']
cat_features = ['tipo_climatico']

In [11]:
#treat null values
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [12]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

In [13]:
#Final pipeline (preproc and RandomForest)
full_pipeline_balanced = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        random_state=42,
        n_estimators=100,
        class_weight='balanced'
    ))
])

In [14]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit

param_dist = {
    'classifier__n_estimators': [100, 200, 300, 500],
    'classifier__max_depth': [None, 5, 10, 15, 20],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2']
}

tscv = TimeSeriesSplit(n_splits=5)

search = RandomizedSearchCV(
    full_pipeline_balanced,
    param_distributions=param_dist,
    n_iter=30,
    cv=tscv,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
print("Best params", search.best_params_)
print("Best score (CV):", search.best_score_)

best_model = search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
print(classification_report(y_test, y_pred_tuned))

Best params {'classifier__n_estimators': 500, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 2, 'classifier__max_features': 'sqrt', 'classifier__max_depth': None}
Best score (CV): 0.4308244521530106
               precision    recall  f1-score   support

   heavy_rain       0.47      0.57      0.51       224
   light_rain       0.31      0.34      0.32       305
moderate_rain       0.22      0.16      0.19       174
      no_rain       0.81      0.78      0.80      1053

     accuracy                           0.62      1756
    macro avg       0.45      0.46      0.45      1756
 weighted avg       0.62      0.62      0.62      1756



In [15]:
print("=== 📊 OVERALL MODEL PERFORMANCE ===")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred_tuned):.2%}\n")
print(classification_report(y_test, y_pred_tuned))

=== 📊 OVERALL MODEL PERFORMANCE ===
Overall Accuracy: 61.67%

               precision    recall  f1-score   support

   heavy_rain       0.47      0.57      0.51       224
   light_rain       0.31      0.34      0.32       305
moderate_rain       0.22      0.16      0.19       174
      no_rain       0.81      0.78      0.80      1053

     accuracy                           0.62      1756
    macro avg       0.45      0.46      0.45      1756
 weighted avg       0.62      0.62      0.62      1756



In [ ]:
# CSV file for the last 3 days + today (necessary data to make the forecasting)
df_ultimos_dias = pd.read_csv('recent_data.csv', sep=';', decimal=',', encoding='utf-8')

df_ultimos_dias['date'] = pd.to_datetime(df_ultimos_dias['Data Medicao'])
df_ultimos_dias = df_ultimos_dias.sort_values('date').reset_index(drop=True)
df_ultimos_dias['tipo_climatico'] = df_ultimos_dias[col_precip].apply(classificar_tempo)
#test

In [17]:
def montar_features_previsao(df_historico, best_model):
    """
    df_historico: DataFrame com os dados mais recentes, já no mesmo formato
    do df original (mesmas colunas, ordenado por data), incluindo pelo menos
    os últimos 3 dias (para as rolling features) + o dia de hoje.
    """
    df_historico = df_historico.sort_values('date').reset_index(drop=True)
    
    df_historico['temp_max_mean_3d'] = df_historico['TEMPERATURA MAXIMA, DIARIA (AUT)(°C)'].rolling(window=3).mean()
    df_historico['precipitation_sum_3d'] = df_historico['PRECIPITACAO TOTAL, DIARIO (AUT)(mm)'].rolling(window=3).sum()
    df_historico['wind_mean_3d'] = df_historico['VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)'].rolling(window=3).mean()
    df_historico['month'] = df_historico['date'].dt.month
    
    linha_hoje = df_historico.iloc[[-1]][features]
    
    return linha_hoje

linha_hoje = montar_features_previsao(df_ultimos_dias, best_model)
previsao_amanha = best_model.predict(linha_hoje)

In [18]:
probabilidades = best_model.predict_proba(linha_hoje)
classes = best_model.named_steps['classifier'].classes_
print("---- PROBABILITIES ----\n")
for classe, prob in zip(classes, probabilidades[0]):
    print(f"{classe}: {prob:.1%}")

---- PROBABILITIES ----

heavy_rain: 20.3%
light_rain: 24.1%
moderate_rain: 19.6%
no_rain: 35.9%
